# 06 · RAG Agent：讓 agent 查得到事實

LLM 會用一樣自信的語氣瞎掰它沒學過的事實。**RAG（檢索增強生成）** 治這個病：別讓模型憑記憶答，**先檢索**外部知識庫、**再根據資料**回答。

## 1. 載入模型 + embedding 模型

多語 embedding 模型 CPU 就能跑。

In [ ]:
%pip install -q -U "transformers>=4.45" accelerate bitsandbytes
%pip install -q sentence-transformers

In [ ]:
import json, re
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"   # 想更強可換 Qwen2.5-3B-Instruct，T4 也裝得下

_bnb = BitsAndBytesConfig(
    load_in_4bit=True,                    # 4-bit 量化：模型體積砍約 75%
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=_bnb, device_map="auto", torch_dtype=torch.float16,
)
model.eval()
print(f"已載入 {MODEL_ID}（4-bit）於 {model.device}")


@torch.no_grad()
def chat(messages, max_new_tokens=512, temperature=0.0):
    """LLM 的唯一介面：丟一串 messages，回模型吐的純文字。

    messages = [{"role": "system"|"user"|"assistant", "content": "..."}]
    temperature=0 → 貪婪解碼（穩定、可重現），>0 → 取樣（多樣）。
    整條軌道只透過這個函式碰模型；要換成別的模型/API，改這裡就好。
    """
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    gen = dict(max_new_tokens=max_new_tokens, pad_token_id=tokenizer.eos_token_id)
    if temperature > 0:
        gen.update(do_sample=True, temperature=temperature, top_p=0.9)
    else:
        gen.update(do_sample=False)
    out = model.generate(**inputs, **gen)
    reply = out[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(reply, skip_special_tokens=True).strip()

## 2. 內嵌的小知識庫

刻意放帶具體數字的事實——小模型容易記錯。

In [ ]:
# 內嵌知識庫：每張卡片一段事實短文（真實世界資料，零下載）。
KB = [
    "玉山是台灣最高峰，主峰海拔 3952 公尺，位於玉山國家公園，橫跨南投、嘉義、高雄。",
    "台灣最長的河流是濁水溪，全長約 186.6 公里，發源於合歡山，向西流入台灣海峽。",
    "日月潭是台灣最大的天然湖泊，位於南投縣魚池鄉，潭面海拔約 748 公尺。",
    "台灣本島面積約 36,197 平方公里，是世界第 38 大島嶼，南北長約 394 公里。",
    "台北101 樓高 508 公尺、地上 101 層，2004 年完工，曾是世界第一高樓直到 2010 年。",
    "台灣高鐵全長約 350 公里，連接南港與左營，最高營運時速 300 公里，2007 年通車。",
    "台灣的氣候北部為副熱帶、南部為熱帶，每年 5 到 6 月有梅雨季，夏秋多颱風。",
    "澎湖群島由約 90 座島嶼組成，位於台灣海峽中央，以玄武岩柱狀節理地形聞名。",
    "台灣原住民族目前官方認定有 16 族，人口約佔總人口的 2.5%，阿美族是人數最多的一族。",
    "蘭嶼位於台東外海，是達悟族（雅美族）的居住地，以拼板舟與飛魚文化著稱。",
]

## 3. 手刻 embedding 檢索

把每張卡片與問題轉成向量，算**餘弦相似度**取最相近的 top-k。向量正規化後，內積就是餘弦相似度。

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
kb_vecs = embedder.encode(KB, normalize_embeddings=True)   # 每張卡片一個向量


def retrieve(query, k=2):
    q = embedder.encode([query], normalize_embeddings=True)[0]
    scores = kb_vecs @ q                                   # 餘弦相似度
    top = np.argsort(-scores)[:k]
    return [KB[i] for i in top]


print(retrieve("台灣最高的山多高？"))

## 4. 對照：沒 RAG vs 有 RAG

In [ ]:
q = "玉山主峰的海拔到底是幾公尺？"

# 沒 RAG：模型憑記憶，可能瞎掰
print("【沒 RAG】", chat([{"role": "user", "content": q}]))

In [ ]:
# 有 RAG：先檢索，再要求模型根據資料作答並標注出處
docs = retrieve(q)
ctx = "\n".join(f"[{i+1}] {d}" for i, d in enumerate(docs))
prompt = f"""只根據以下資料回答問題，並標注你用了哪一條（例如 [1]）。
若資料中沒有答案，就說不知道。

資料：
{ctx}

問題：{q}"""
print("【有 RAG】", chat([{"role": "user", "content": prompt}]))

## 5. 把檢索包成工具，接回 ReAct

不是每題都要查。把 `retrieve` 當成一支工具，讓 agent **自己決定何時查**——就是第 04 課的工具路由，多註冊一支 `retrieve` 而已。

In [ ]:
# 概念示意：retrieve 就是一支會回傳文件字串的工具
def retrieve_tool(query: str) -> str:
    return " | ".join(retrieve(query, k=2))


print(retrieve_tool("澎湖的地形特色"))
# 接回第 04 課的 REGISTRY，agent 即可在需要時自己呼叫 retrieve。

## 小結

- **RAG = 先檢索、再 grounding**，把「憑記憶」換成「靠資料」，治幻覺、附出處。
- 檢索靠 **embedding 餘弦相似度**，幾行 numpy 手刻。
- 把 `retrieve` 包成工具接回 ReAct，agent 就會**自己決定何時查資料**。

下一課：複雜任務拆給**多個代理**協作。